In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_firm_structure_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "FirmStructureId string,"
        #     "SourceSystem string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "FirmStructureKey string,"
        #     "EmployeePartyKey string,"
        #     "EmployeeCategory string,"
        #     "EmployeeFirstName string,"
        #     "EmployeeLastName string,"
        #     "EmployeeFullName string,"
        #     "EmployeeEmail string,"
        #     "EmployeeUsername string,"
        #     "EmployeeUserTypeCode string,"
        #     "EmployeeUserType string,"
        #     "EmployeePositionCode string ,"
        #     "EmployeePosition string,"
        #     "DepartmentCode string,"
        #     "Department string,"
        #     "TeamCode string,"
        #     "Team string,"
        #     "SupervisorPartyKey string,"
        #     "SupervisorPartyName string,"
        #     "SupervisorUsername string,"
        #     "ManagerPartyKey string,"
        #     "ManagerPartyName string,"
        #     "ManagerUsername string,"
        #     "DirectorPartyKey string,"
        #     "DirectorPartyName string,"
        #     "DirectorUsername string,"
        #     "IsEmployeeActiveFlag boolean,"
        #     "CreatedDate date,"
        #     "LastUpdatedDate date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.FirmStructure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== employees ==================

        employees_extra_filters = None

        if employees_extra_filters is not None:
            employees_extra_filters.append(date_filter)
            employees_final_filters = " AND ".join(employees_extra_filters)
        else:
            employees_final_filters = date_filter

        employees_selected_fields = [
            "emp_Code",
            "emp_first_name",
            "emp_last_name",
            "emp_login",
            "emp_user_type",
            "emp_job_desc",
            "dep_code",
            "department",
            "team_id",
            "team_name",
            "supervisor_emp_code",
            "supervisor_login",
            "manager_emp_code",
            "manager_login",
            "emp_active",
            "created",
            "updated",
            "businessdate",
        ]

        employees = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_employees",
            employees_final_filters,
            employees_selected_fields,
        )

        # ================== external_companies ======================
        external_companies_extra_filters = None

        if external_companies_extra_filters is not None:
            external_companies_extra_filters.append(date_filter)
            external_companies_final_filters = " AND ".join(
                external_companies_extra_filters
            )
        else:
            external_companies_final_filters = date_filter

        external_companies_selected_fields = [
            "extc_code",
            "extc_name",
            "extc_intcode",
            "EXTC_COMMENT",
            "BusinessDate",
        ]

        external_companies = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_external_companies",
            external_companies_final_filters,
            external_companies_selected_fields,
        )

        # ================== lawyer ==================

        lawyer_extra_filters = None

        if lawyer_extra_filters is not None:
            lawyer_extra_filters.append(date_filter)
            lawyer_final_filters = " AND ".join(lawyer_extra_filters)
        else:
            lawyer_final_filters = date_filter

        lawyer_selected_fields = [
            "LAWY_CODE",
            "LAWY_LAST_NAME",
            "LAWY_FIRST_NAME",
            "LAWY_ACTIVE_FLAG",
            "LAWY_CREATED",
            "BusinessDate",
        ]

        lawyer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lawyer",
            lawyer_final_filters,
            lawyer_selected_fields,
        )

        # ================== firm_structure_df1 ==================
        firm_structure_qc_primary_key_fields_1 = [
            employees.emp_Code,
            lit("Employee"),
            lit("qc"),
        ]
        FirmStructureKey_qc_primary_key_fields_1 = [employees.emp_Code, lit("Employee")]

        sup = employees.select(
            col("emp_Code").cast("string").alias("sup_code"),
            col("businessdate").cast("date").alias("sup_bd"),
            concat_ws(
                " ",
                coalesce(col("emp_first_name"), lit("")),
                coalesce(col("emp_last_name"), lit("")),
            ).alias("sup_full_name"),
        ).alias("sup")

        mgr = employees.select(
            col("emp_Code").cast("string").alias("mgr_code"),
            col("businessdate").cast("date").alias("mgr_bd"),
            concat_ws(
                " ",
                coalesce(col("emp_first_name"), lit("")),
                coalesce(col("emp_last_name"), lit("")),
            ).alias("mgr_full_name"),
        ).alias("mgr")

        firm_structure_qc_1 = (
            (
                employees.withColumn(
                    "FirmStructureId",
                    sha2(
                        concat_ws(";", *firm_structure_qc_primary_key_fields_1),
                        256,
                    ),
                )
            )
            .withColumn(
                "FirmStructureKey",
                concat_ws(";", *FirmStructureKey_qc_primary_key_fields_1),
            )
            .withColumn(
                "EmployeeFullName",
                concat_ws(
                    " ",
                    coalesce(employees["emp_first_name"], lit("")),
                    coalesce(employees["emp_last_name"], lit("")),
                ),
            )
            .join(
                sup,
                (col("supervisor_emp_code").cast("string") == col("sup.sup_code"))
                & (col("businessdate").cast("date") == col("sup.sup_bd")),
                "left",
            )
            .join(
                mgr,
                (col("manager_emp_code").cast("string") == col("mgr.mgr_code"))
                & (col("businessdate").cast("date") == col("mgr.mgr_bd")),
                "left",
            )
        )
        
        

        firm_structure_qc_1 = firm_structure_qc_1.select(
            firm_structure_qc_1.FirmStructureId.alias("FirmStructureId"),
            lit("qc").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_qc_1.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            employees.emp_Code.cast("string").alias("EmployeePartyKey"),
            lit("Employee").alias("EmployeeCategory"),
            employees.emp_first_name.cast("string").alias("EmployeeFirstName"),
            employees.emp_last_name.cast("string").alias("EmployeeLastName"),
            firm_structure_qc_1.EmployeeFullName.cast("string").alias(
                "EmployeeFullName"
            ),
            lit(None).cast("string").alias("EmployeeEmail"),
            employees.emp_login.cast("string").alias("EmployeeUsername"),
            employees.emp_user_type.cast("string").alias("EmployeeUserTypeCode"),
            employees.emp_user_type.cast("string").alias("EmployeeUserType"),
            employees.emp_job_desc.cast("string").alias("EmployeePositionCode"),
            employees.emp_job_desc.cast("string").alias("EmployeePosition"),
            employees.dep_code.cast("string").alias("DepartmentCode"),
            employees.department.cast("string").alias("Department"),
            employees.team_id.cast("string").alias("TeamCode"),
            employees.team_name.cast("string").alias("Team"),
            employees.supervisor_emp_code.cast("string").alias("SupervisorPartyKey"),
            col("sup.sup_full_name").cast("string").alias("SupervisorPartyName"),   
            employees.supervisor_login.cast("string").alias("SupervisorUsername"),
            employees.manager_emp_code.cast("string").alias("ManagerPartyKey"),
            col("mgr.mgr_full_name").cast("string").alias("ManagerPartyName"),
            employees.manager_login.cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            employees.emp_active.cast("boolean").alias("IsEmployeeActiveFlag"),
            employees.created.cast("date").alias("CreatedDate"),
            employees.updated.cast("date").alias("LastUpdatedDate"),
            employees.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== firm_structure_df2 ==================
        firm_structure_qc_primary_key_fields_2 = [
            external_companies.extc_code,
            lit("External Company"),
            lit("qc"),
        ]
        FirmStructureKey_qc_primary_key_fields_2 = [
            external_companies.extc_code,
            lit("External Company"),
        ]

        firm_structure_qc_2 = (
            external_companies.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_qc_primary_key_fields_2),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_qc_primary_key_fields_2),
        )

        firm_structure_qc_2 = firm_structure_qc_2.select(
            firm_structure_qc_2.FirmStructureId.alias("FirmStructureId"),
            lit("qc").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_qc_2.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            external_companies.extc_code.cast("string").alias("EmployeePartyKey"),
            lit("External Company").alias("EmployeeCategory"),
            lit(None).cast("string").alias("EmployeeFirstName"),
            lit(None).cast("string").alias("EmployeeLastName"),
            external_companies.extc_name.cast("string").alias("EmployeeFullName"),
            lit(None).cast("string").alias("EmployeeEmail"),
            lit(None).cast("string").alias("EmployeeUsername"),
            external_companies.extc_intcode.cast("string").alias(
                "EmployeeUserTypeCode"
            ),
            external_companies.EXTC_COMMENT.cast("string").alias("EmployeeUserType"),
            lit(None).cast("string").alias("EmployeePositionCode"),
            lit(None).cast("string").alias("EmployeePosition"),
            lit(None).cast("string").alias("DepartmentCode"),
            lit(None).cast("string").alias("Department"),
            lit(None).cast("string").alias("TeamCode"),
            lit(None).cast("string").alias("Team"),
            lit(None).cast("string").cast("string").alias("SupervisorPartyKey"),
            lit(None).cast("string").alias("SupervisorPartyName"),
            lit(None).cast("string").alias("SupervisorUsername"),
            lit(None).cast("string").alias("ManagerPartyKey"),
            lit(None).cast("string").alias("ManagerPartyName"),
            lit(None).cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            lit(None).cast("boolean").alias("IsEmployeeActiveFlag"),
            lit(None).cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("LastUpdatedDate"),
            external_companies.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ================== firm_structure_df3 ==================
        firm_structure_qc_primary_key_fields_3 = [
            lawyer.LAWY_CODE,
            lit("Lawyer"),
            lit("qc"),
        ]
        FirmStructureKey_qc_primary_key_fields_3 = [lawyer.LAWY_CODE, lit("Lawyer")]

        firm_structure_qc_3 = (
            lawyer.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(";", *firm_structure_qc_primary_key_fields_3),
                    256,
                ),
            )
        ).withColumn(
            "FirmStructureKey",
            concat_ws(";", *FirmStructureKey_qc_primary_key_fields_3),
        )

        firm_structure_qc_3 = firm_structure_qc_3.select(
            firm_structure_qc_3.FirmStructureId.alias("FirmStructureId"),
            lit("qc").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_qc_3.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            lawyer.LAWY_CODE.cast("string").alias("EmployeePartyKey"),
            lit("Lawyer").alias("EmployeeCategory"),
            lit(None).cast("string").alias("EmployeeFirstName"),
            lit(None).cast("string").alias("EmployeeLastName"),
            lawyer.LAWY_LAST_NAME.cast("string").alias("EmployeeFullName"),
            lit(None).cast("string").alias("EmployeeEmail"),
            lit(None).cast("string").alias("EmployeeUsername"),
            lit(None).cast("string").alias("EmployeeUserTypeCode"),
            lawyer.LAWY_FIRST_NAME.cast("string").alias("EmployeeUserType"),
            lit(None).cast("string").alias("EmployeePositionCode"),
            lit(None).cast("string").alias("EmployeePosition"),
            lit(None).cast("string").alias("DepartmentCode"),
            lit(None).cast("string").alias("Department"),
            lit(None).cast("string").alias("TeamCode"),
            lit(None).cast("string").alias("Team"),
            lit(None).cast("string").cast("string").alias("SupervisorPartyKey"),
            lit(None).cast("string").alias("SupervisorPartyName"),
            lit(None).cast("string").alias("SupervisorUsername"),
            lit(None).cast("string").alias("ManagerPartyKey"),
            lit(None).cast("string").alias("ManagerPartyName"),
            lit(None).cast("string").alias("ManagerUsername"),
            lit(None).cast("string").alias("DirectorPartyKey"),
            lit(None).cast("string").alias("DirectorPartyName"),
            lit(None).cast("string").alias("DirectorUsername"),
            lawyer.LAWY_ACTIVE_FLAG.cast("boolean").alias("IsEmployeeActiveFlag"),
            lawyer.LAWY_CREATED.cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("LastUpdatedDate"),
            lawyer.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ========================== Append ==========================

        Result_df = firm_structure_qc_1.union(firm_structure_qc_2).union(
            firm_structure_qc_3
        )
        Result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_firm_structure_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)